# Week 1 - Video -> Browsable 3D Room (Person A pipeline)
**Notebook settings (right sidebar):** Accelerator = *GPU T4 x2* - Internet = *On* - Persistence = *Files only*.

Each stage below is a **separate process** (`!python ...`) so VRAM is released after every stage (PDF 4.3).
Run the cells top to bottom the first time. Re-runs can skip setup.

In [10]:
# 0. Get the code. EITHER clone your repo ...
#REPO = "https://github.com/<your-team>/<your-repo>.git"      # <- edit
# !git clone $REPO /kaggle/working/repo
# ... OR attach the repo as a Kaggle Dataset and copy it:
!cp -r /kaggle/input/datasets/shahdelsawy/pipeline/week1_pipeline /kaggle/working/
%cd /kaggle/working/week1_pipeline

/kaggle/working/week1_pipeline


In [3]:
# 1. One-time setup: ffmpeg, VGGT, COLMAP(+GLOMAP), gsplat.  ~10-20 min the first time (gsplat compiles CUDA on first use).
!bash scripts/setup_kaggle.sh all


=== base python deps ===
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers

=== VGGT ===
Cloning into '/kaggle/working/vggt'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 199 (delta 6), reused 187 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 62.06 MiB | 35.17 MiB/s, done.
Resolving deltas: 100% (6/6), done.
VGGT import OK, torch 2.11.0+cu128

=== COLMAP (+ GLOMAP / global_mapper) ===
glomap: /kaggle/working/tools/glomap
colmap: /kaggle/working/tools/colmap  (0 global_mapper entries)
python (must stay the system one): /usr/local/bin/python

=== gsplat + examples ===
python: /usr/local/bin/python
gsplat version: 1.5.3
Note: switching to '937e29912570c372bed6747a5c9bf85fed877bae'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
stat

In [4]:
# 2. Environment for this session
import os, re, shutil
for line in open("/kaggle/working/env.sh"):
    m = re.match(r"export (\w+)=(.*)", line.strip())
    if m: os.environ[m.group(1)] = os.path.expandvars(m.group(2))
# safety: the conda env's bare python must never shadow the system python
os.environ["PATH"] = ":".join(x for x in os.environ["PATH"].split(":") if x and "/kaggle/working/cm/bin" not in x and "$PATH" not in x)
print("python :", shutil.which("python"), "| colmap:", shutil.which("colmap"), "| glomap:", shutil.which("glomap"))
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"          # T4
os.environ["W1_ROOT"] = "/kaggle/working/workspace"   # everything is written here (persisted in Output)
os.environ["W1_PROJECT"] = "demo"
VIDEO = "/kaggle/input/datasets/shahdelsawy/room-video/video.mp4"    # <- edit: 2-5 min, one room, slow motion
!nvidia-smi --query-gpu=name,memory.total --format=csv

python : /usr/local/bin/python | colmap: /kaggle/working/tools/colmap | glomap: /kaggle/working/tools/glomap
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## Stage 1 - frames (ffmpeg, 1-4 fps, long side <= 960, blur filter)

In [5]:
!python stages/s1_extract_frames.py --video $VIDEO --fps auto --long-side 960

[progress] extracting frames: 0.0%
[progress] extracting frames: 0.8%
[progress] extracting frames: 2.2%
[progress] extracting frames: 7.5%
[progress] extracting frames: 12.0%
[s1] 61 frames @ 1.0 fps -> kept 61 (0 blurry dropped), size 960x540
[progress] extracting frames: 15.0%


## Stage 2 - poses. Run BOTH for the required VGGT-vs-GLOMAP experiment (same frames)

In [6]:
# VGGT: fp16 on T4, chunks of 32 frames with 8 shared frames; also measure VRAM vs frames-per-chunk
!python stages/s2a_poses_vggt.py --chunk-size 32 --overlap 8 --memory-sweep 8,16,32,48

[progress] estimating poses: 15.0%
[vggt] device=cuda dtype=torch.float16 frames=61
config.json: 100%|████████████████████████████| 62.0/62.0 [00:00<00:00, 340kB/s]

model.safetensors: downloading bytes: ▏                     | 52.5MB, 3.54MB/s  
model.safetensors: downloading bytes: ▎                     | 83.9MB, 6.29MB/s  
model.safetensors: downloading bytes: ▉                     |  208MB, 17.1MB/s  
model.safetensors: downloading bytes: █▎                    |  311MB, 25.6MB/s  
model.safetensors: downloading bytes: █▋                    |  384MB, 31.4MB/s  
model.safetensors: downloading bytes: █▊                    |  427MB, 34.2MB/s  
model.safetensors: downloading bytes: ██▎                   |  518MB, 41.7MB/s  
model.safetensors: downloading bytes: ██▍                   |  555MB, 45.2MB/s  
model.safetensors: downloading bytes: ██▊                   |  643MB, 51.9MB/s  
model.safetensors: downloading bytes: ██▉                   |  683MB, 55.4MB/s  
model.safetensors: downl

In [11]:
%%bash
W=/kaggle/working
$W/micromamba install -y -q -p $W/cm -c conda-forge libfaiss 2>&1 | tail -3
for tool in colmap glomap; do
  printf '#!/bin/sh\nexport LD_LIBRARY_PATH=%s/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}\nexec %s/bin/%s "$@"\n' "$W/cm" "$W/cm" "$tool" > $W/tools/$tool
  chmod +x $W/tools/$tool
done
export PATH=$W/tools:$PATH
colmap help 2>&1 | head -2
export LD_LIBRARY_PATH=$W/cm/lib
ldd $W/cm/bin/colmap | grep "not found" || echo "colmap: no missing libraries"
exit 0

COLMAP 3.8 -- Structure-from-Motion and Multi-View Stereo
              (Commit Unknown on Unknown without CUDA)
colmap: no missing libraries


In [13]:
%%bash
L=/kaggle/working/workspace/demo/metrics/logs/glomap_mapper.log
echo "--- first 25 lines"; head -25 $L
echo "--- error lines"; grep -n -i "check failed\|invalid\|error\|terminate" $L | head -10
echo "--- versions"; colmap help 2>&1 | head -3; glomap -h 2>&1 | head -2
echo "--- retriangulation flag"; glomap mapper -h 2>&1 | grep -i -B1 -A1 "retri"
exit 0

--- first 25 lines
$ /kaggle/working/tools/glomap mapper --database_path /kaggle/working/workspace/demo/poses/glomap/_work/database.db --image_path /kaggle/working/workspace/demo/frames/images --output_path /kaggle/working/workspace/demo/poses/glomap/_work/sparse
 Loading Images 61 / 61
 Loading Image Pair 679 / 679
I20261101 20:59:34.189051 139855539160960 colmap_converter.cc:424] Pairs read done. 0 / 679 are invalid
I20261101 20:59:34.189495 139855539160960 global_mapper.cc:84] Loaded database
-------------------------------------
Running preprocessing ...
-------------------------------------
I20261101 20:59:34.189706 139855539160960 view_graph_manipulation.cc:255] Decompose relative pose for 0 pairs
I20261101 20:59:34.190677 139855539160960 view_graph_manipulation.cc:309] Decompose relative pose done. 0 pairs are pure rotation
I20261101 20:59:34.190809 139855539160960 timer.cc:86] Elapsed time: 0.00127 [seconds]
-------------------------------------
Running view graph calibration .

In [14]:
%%bash
W=/kaggle/working/workspace/demo/poses/glomap
mkdir -p $W/_work/sparse_retry
glomap mapper --database_path $W/_work/database.db \
  --image_path /kaggle/working/workspace/demo/frames/images \
  --output_path $W/_work/sparse_retry --skip_retriangulation 1 2>&1 | tail -15
echo "--- output:"; ls -R $W/_work/sparse_retry | head
exit 0

I20261101 21:05:51.104721 137244025337728 bundle_adjustment.cc:103] Ceres Solver Report: Iterations: 3, Initial cost: 3.697236e+04, Final cost: 3.696929e+04, Termination: CONVERGENCE
I20261101 21:05:51.104815 137244025337728 global_mapper.cc:224] Global bundle adjustment iteration 3 / 3, stage 2 finished
I20261101 21:05:51.106030 137244025337728 image_undistorter.cc:20] Undistorting images..
I20261101 21:05:51.117872 137244025337728 image_undistorter.cc:44] Image undistortion done
I20261101 21:05:51.118578 137244025337728 global_mapper.cc:238] Filtering tracks by reprojection ...
I20261101 21:05:51.128452 137244025337728 track_filter.cc:49] Filtered 762 / 10893 tracks by reprojection error
I20261101 21:05:51.171134 137244025337728 image_undistorter.cc:20] Undistorting images..
I20261101 21:05:51.181651 137244025337728 image_undistorter.cc:44] Image undistortion done
I20261101 21:05:51.182336 137244025337728 global_mapper.cc:264] Filtering tracks by reprojection ...
I20261101 21:05:51.1

In [15]:
%%bash
cd /kaggle/working/week1_pipeline
sed -i 's/be\["mapper"\] + \["--database_path"/be["mapper"] + ["--skip_retriangulation", "1", "--database_path"/' stages/s2b_poses_glomap.py
grep -n "skip_retriangulation" stages/s2b_poses_glomap.py
exit 0

170:            t_map = sh(be["mapper"] + ["--skip_retriangulation", "1", "--database_path", str(db), "--image_path", str(imgs),


In [16]:
# GLOMAP: features -> sequential matching -> global mapper.  (skip this cell if no global mapper installed -> use --method vggt)
!python stages/s2b_poses_glomap.py --overlap 12 --no-gpu

[progress] estimating poses: 15.0%
[glomap] feature extraction: 28.5s
[progress] estimating poses: 22.5%
[glomap] sequential matching: 105.3s
[progress] estimating poses: 30.0%
[glomap] glomap mapper: 40.0s
[progress] estimating poses: 39.0%
[glomap] registered 61/61 frames, 10555 points (glomap mapper)
[progress] estimating poses: 45.0%


In [17]:
# Comparison report + pick the method used for the project (writes scene/)
!python stages/s3_compare_poses.py --choose auto
from IPython.display import Markdown, Image, display
display(Markdown(open(os.environ["W1_ROOT"] + "/demo/metrics/pose_comparison.md").read()))

# VGGT vs GLOMAP

Frames fed to both methods: **61**

| method | registered | runtime (s) | peak VRAM (MiB) | points3D | path length |
|---|---|---|---|---|---|
| vggt | 61 | 114.17 | 11289.0 | 300,000 | 3.212 |
| glomap | 61 | 176.05 | 3.0 | 10,555 | 34.025 |

Trajectory agreement (Sim3-aligned, 61 common frames): RMSE = 0.71% of scene extent.

![visual check](pose_comparison.png)

## Draft justification (edit before submitting)

We chose GLOMAP for the project because both register all frames and runtimes are comparable, so the more accurate classical baseline wins. VGGT: 61 frames, 114s, 11289 MiB peak VRAM. GLOMAP: 61 frames, 176s, 3 MiB peak VRAM.
[compare] scene/ <- glomap


# VGGT vs GLOMAP

Frames fed to both methods: **61**

| method | registered | runtime (s) | peak VRAM (MiB) | points3D | path length |
|---|---|---|---|---|---|
| vggt | 61 | 114.17 | 11289.0 | 300,000 | 3.212 |
| glomap | 61 | 176.05 | 3.0 | 10,555 | 34.025 |

Trajectory agreement (Sim3-aligned, 61 common frames): RMSE = 0.71% of scene extent.

![visual check](pose_comparison.png)

## Draft justification (edit before submitting)

We chose GLOMAP for the project because both register all frames and runtimes are comparable, so the more accurate classical baseline wins. VGGT: 61 frames, 114s, 11289 MiB peak VRAM. GLOMAP: 61 frames, 176s, 3 MiB peak VRAM.


## Stage 4 - 3D Gaussian Splatting (quick = 7k iters to confirm, then full = 30k)

In [19]:
%%bash
touch /kaggle/working/gsplat/examples/datasets/__init__.py
ls /kaggle/working/gsplat/examples/datasets/
cd /kaggle/working/week1_pipeline
python /kaggle/working/gsplat/examples/simple_trainer.py default --help 2>&1 | grep -i -E "data[-_]dir|steps[-_]scaler|max[-_]steps" | head -5
exit 0

colmap.py
download_dataset.py
__init__.py
normalize.py
traj.py


In [20]:
%%bash
cd /kaggle/working/gsplat/examples
python simple_trainer.py default --help 2>&1 | tail -25
exit 0

Traceback (most recent call last):
  File "/kaggle/working/gsplat/examples/simple_trainer.py", line 18, in <module>
    from datasets.colmap import Dataset, Parser
  File "/kaggle/working/gsplat/examples/datasets/colmap.py", line 10, in <module>
    from pycolmap import SceneManager
  File "/usr/local/lib/python3.13/dist-packages/pycolmap/__init__.py", line 4, in <module>
    from .scene_manager import SceneManager
  File "/usr/local/lib/python3.13/dist-packages/pycolmap/scene_manager.py", line 21, in <module>
    class SceneManager:
    ...<648 lines>...
                    self.scene_graph[image_id2][image_id1] += 1
  File "/usr/local/lib/python3.13/dist-packages/pycolmap/scene_manager.py", line 22, in SceneManager
    INVALID_POINT3D = np.uint64(-1)
OverflowError: Python integer -1 out of bounds for uint64


In [21]:
%%bash
F=/usr/local/lib/python3.13/dist-packages/pycolmap/scene_manager.py
sed -i 's/np.uint64(-1)/np.uint64(np.iinfo(np.uint64).max)/' $F
grep -n "INVALID_POINT3D = " $F
cd /kaggle/working/gsplat/examples
python simple_trainer.py default --help 2>&1 | tail -12
exit 0

22:    INVALID_POINT3D = np.uint64(np.iinfo(np.uint64).max)
│       False)                                                                 │
│ --strategy.revised-opacity, --strategy.no-revised-opacity                    │
│       Whether to use revised opacity heuristic from arXiv:2404.06109         │
│       (experimental). Default is False. (default: False)                     │
│ --strategy.verbose, --strategy.no-verbose                                    │
│       Whether to print verbose information. Default is False. (default:      │
│       True)                                                                  │
│ --strategy.key-for-gradient {means2d,gradient_2dgs}                          │
│       Which variable uses for densification strategy.                        │
│       3DGS uses "means2d" gradient and 2DGS uses a similar gradient which    │
│       stores in variable "gradient_2dgs". (default: means2d)                 │
╰────────────────────────────────────────────────

In [22]:
!python stages/s4_train_3dgs.py --trainer gsplat --mode quick

[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.4%
[progress] training: 45.4%
[progress] training: 45.5%
[progress] training: 45.5%
[progress] training: 45.6%
[progress] training: 45.6%
[progress] training: 45.7%
[progress] training: 45.7%
[progress] training: 45.8%
[progress] training: 45.8%
[progress] training: 45.9%
[progress] training: 45.9%
[progress] training: 46.0%
[progress] training: 46.0%
[progress] training: 46.1%
[progress] training: 46.1%
[progress] training: 46.2%
[progress] training: 46.2%
[progress] training: 46.3%
[progress] training: 46.3%
[progress] training: 46.4%
[progress] training: 46.4%
[progress] training: 46.5%
[progress] training: 46.5%
[progress] training: 46.5%
[progress] training: 46.6%
[progress] training: 46.6%
[progress] training: 46.7%
[

In [23]:
# quality run (about 30k iters; check the quick run looked sane first!)
!python stages/s4_train_3dgs.py --trainer gsplat --mode full

[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.0%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.1%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.2%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.3%
[progress] training: 45.4%
[progress] training: 45.4%
[progress] training: 45.4%
[progress] training: 45.4%
[

## Stage 5 - export for the web viewer, then the generated deliverables

In [24]:
!python stages/s5_export_splat.py
!python tools/make_reports.py
!cat docs/generated/VRAM_RUNTIME_TABLE.md

[progress] exporting: 90.0%
[progress] exporting: 93.0%
[progress] exporting: 97.0%
[export] 1,054,118 Gaussians; ply 237.25 MB, splat 32.17 MB
[progress] exporting: 100.0%
wrote docs/generated/VRAM_RUNTIME_TABLE.md, docs/generated/WEEK2_HANDOFF.md, /kaggle/working/workspace/demo/manifest.json
# VRAM and runtime per stage

| stage | wall time (s) | peak VRAM, nvidia-smi (MiB) | peak VRAM, torch alloc (MiB) | notes |
|---|---|---|---|---|
| 1. Frame extraction (ffmpeg + blur filter) | 132.2 | 3 | 0 | frames_kept=61 |
| 2a. Poses: VGGT (chunked, fp16) | 114.2 | 11289 | 9591 | registered_frames=61, chunks=3, dtype=torch.float16 |
| 2b. Poses: GLOMAP (features + seq. matching + mapper) | 176.1 | 3 | 0 | registered_frames=61 |
| 4. 3DGS training | 2550.5 | 3609 | 0 | num_gaussians=1496393, iters=30000 |
| 5. Export (.ply -> .splat) | 2.9 | 3 | 0 | num_gaussians=1054118, splat_mb=32.17 |

## VGGT peak VRAM vs frames per chunk

dtype `torch.float16`, input [518, 294]

| frames | seconds | pea

In [25]:
# 6. Persist what matters from /kaggle/working before the session ends (PDF 4.3)
!cd $W1_ROOT/demo && zip -qr /kaggle/working/demo_export.zip export manifest.json metrics scene frames/frames_index.json
!ls -lh /kaggle/working/demo_export.zip
# frames + poses + depth (Week 2 inputs) are in $W1_ROOT/demo/{frames,poses,scene}; zip them too if the backend needs them:
# !cd $W1_ROOT/demo && zip -qr /kaggle/working/demo_week2_inputs.zip frames poses scene

-rw-r--r-- 1 root root 232M Nov  1 22:41 /kaggle/working/demo_export.zip


In [26]:
import os

print(os.path.exists('/kaggle/working/demo_export.zip'))

True


In [27]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/demo_export.zip'))

/kaggle/working/demo_export.zip

In [28]:
import os
from IPython.display import FileLink, display

os.chdir('/kaggle/working')

print(os.listdir('.'))

['demo_export.zip', 'tools', 'micromamba', 'gsplat', 'cm', 'workspace', '.virtual_documents', 'bin', 'week1_pipeline', 'vggt', 'env.sh']


In [30]:
display(FileLink('demo_export.zip'))

/kaggle/working/demo_export.zip

### One-shot alternative
```
!python run_pipeline.py --video $VIDEO --project demo --method both --train-mode quick
```
With the backend: add `--backend-url https://<your>.trycloudflare.com --task-id <id>` and the real stage progress is POSTed to `/tasks/{id}/progress`.